In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

# exp02_01 ルートをパスに追加
sys.path.append(os.path.abspath('..'))
from configs.config import *

# 設定

In [ ]:
# ## 実験バージョン（ファイル保存名に使用）
EXP_VER = 'exp02_01'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'model: {MODEL_CHECKPOINT}')

# データ読み込み

`create_prompt.ipynb` を先に実行してください。

In [ ]:
df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prompt_exp02_01_train.pkl'))
df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prompt_exp02_01_test.pkl'))
df_sample_submission = pd.read_csv(os.path.join(DIR_INPUT, FILE_SAMPLE_SUBMISSION))

print(f'train: {df_train.shape}, test: {df_test.shape}')
print(f'positive rate: {df_train["labels"].mean():.4f}')

# トークナイザー・ユーティリティ

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


class TrainTokenizer:
    """学習用トークナイザー（labelsを付与）"""

    def __init__(self, tokenizer: PreTrainedTokenizerBase, max_length: int) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(
            batch['prompt'], max_length=self.max_length, truncation=True
        )
        return {**tokenized, 'labels': batch['labels']}


class TestTokenizer:
    """推論用トークナイザー（labelsなし）"""

    def __init__(self, tokenizer: PreTrainedTokenizerBase, max_length: int) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch: dict) -> dict:
        tokenized = self.tokenizer(
            batch['prompt'], max_length=self.max_length, truncation=True
        )
        return {**tokenized}


def compute_metrics(eval_preds: EvalPrediction) -> dict:
    """AUCとAccuracyを計算する"""
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    auc = roc_auc_score(labels, probs)
    acc = accuracy_score(y_true=labels, y_pred=probs > 0.5)
    return {'auc': auc, 'acc': acc}


train_tokenizer = TrainTokenizer(tokenizer, max_length=MAX_LENGTH)
test_tokenizer = TestTokenizer(tokenizer, max_length=MAX_LENGTH)

# CV学習・推論

In [ ]:
# ## テストデータをトークナイズ（全fold共通）
test_ds = Dataset.from_pandas(df_test[['prompt']].reset_index(drop=True))
test_ds = test_ds.map(test_tokenizer, batched=True)

# ## OOF・テスト予測の格納先
arr_oof_preds = np.zeros(len(df_train))
arr_test_preds = np.zeros(len(df_test))
arr_folds = np.zeros(len(df_train))

# ## StratifiedGroupKFold（社員番号でグループ化、targetで層化）
kfold = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(kfold.split(
    df_train,
    df_train['labels'],
    df_train['社員番号'],
))

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(splits):
    print(f'\n===== Fold {fold + 1} / {N_SPLITS} =====')

    df_tr = df_train.iloc[train_idx].reset_index(drop=True)
    df_va = df_train.iloc[valid_idx].reset_index(drop=True)
    arr_folds[valid_idx] = fold + 1

    # ## stepsの計算（2 epoch分の更新ステップ数）
    steps = max(1, 2 * int(
        len(df_tr) / BATCH_SIZE_TRAIN / GRADIENT_ACCUMULATION_STEPS
    ))

    # ## モデル保存先
    model_dir = os.path.join(
        DIR_MODEL, EXP_VER, f'{MODEL_NAME}_fold{fold + 1}'
    )
    os.makedirs(model_dir, exist_ok=True)

    # ## TrainingArguments
    training_args = TrainingArguments(
        output_dir=model_dir,
        overwrite_output_dir=True,
        report_to='none',
        num_train_epochs=N_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE_TRAIN,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=BATCH_SIZE_EVAL,
        eval_strategy='steps',
        logging_steps=steps,
        do_eval=True,
        eval_steps=steps,
        save_total_limit=1,
        save_strategy='steps',
        save_steps=steps,
        load_best_model_at_end=True,
        optim='adamw_torch',
        bf16=True,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        metric_for_best_model='auc',
        greater_is_better=True,
        seed=SEED,
        data_seed=SEED,
    )

    # ## モデル初期化
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=2,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

    # ## データセット作成
    train_ds = Dataset.from_pandas(df_tr[['prompt', 'labels']])
    valid_ds = Dataset.from_pandas(df_va[['prompt', 'labels']])
    train_ds = train_ds.map(train_tokenizer, batched=True)
    valid_ds = valid_ds.map(train_tokenizer, batched=True)

    # ## Trainer
    trainer = Trainer(
        args=training_args,
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()

    # ## OOF予測
    oof_logits = trainer.predict(valid_ds).predictions
    oof_probs = torch.from_numpy(oof_logits).float().softmax(-1).numpy()[:, 1]
    arr_oof_preds[valid_idx] = oof_probs

    # ## テスト予測（fold平均）
    test_logits = trainer.predict(test_ds).predictions
    test_probs = torch.from_numpy(test_logits).float().softmax(-1).numpy()[:, 1]
    arr_test_preds += test_probs / N_SPLITS

    # ## fold AUC
    fold_auc = roc_auc_score(df_va['labels'], oof_probs)
    print(f'Fold {fold + 1} AUC: {fold_auc:.6f}')

# CV評価

In [ ]:
cv_auc = roc_auc_score(df_train['labels'], arr_oof_preds)
print(f'CV AUC (全体): {cv_auc:.6f}')

# foldごとのAUC
for fold in range(1, N_SPLITS + 1):
    mask = arr_folds == fold
    fold_auc = roc_auc_score(df_train.loc[mask, 'labels'], arr_oof_preds[mask])
    print(f'  fold{fold}: {fold_auc:.6f}')

# OOF保存

In [ ]:
df_oof = df_train[['社員番号', 'category', 'labels']].copy()
df_oof['preds'] = arr_oof_preds
df_oof['fold'] = arr_folds.astype(int)

oof_path = os.path.join(DIR_INTERIM, f'df_oof_{EXP_VER}.csv')
df_oof.to_csv(oof_path, index=False)
print(f'OOF saved: {oof_path}')

# Submission作成

In [ ]:
os.makedirs(DIR_SUBMISSIONS, exist_ok=True)

# ## df_testはcreate_prompt.ipynbでtest.csvの行順を保持しているので、そのまま使用
df_submit = pd.DataFrame({'target': arr_test_preds})

run_name = f'{MODEL_NAME}_{datetime.now().strftime("%Y%m%d%H%M")}'
submit_path = os.path.join(DIR_SUBMISSIONS, f'{run_name}_submission.csv')
df_submit.to_csv(submit_path, header=True, index=False)
print(f'submission saved: {submit_path}')
print(df_submit['target'].describe())